In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Bank Fraud Detection — XGBoost Model
# MAGIC
# MAGIC **Workspace:** Ujjivan (`ujjivan_2.gold`)
# MAGIC **Model type:** XGBoost binary classifier
# MAGIC **Target:** `Is_Fraud`
# MAGIC
# MAGIC This notebook:
# MAGIC 1. Loads transaction features from the gold layer
# MAGIC 2. Splits data into train / validation / test sets
# MAGIC 3. Trains an XGBoost classifier with early stopping and class imbalance handling
# MAGIC 4. Selects a decision threshold based on a target recall
# MAGIC 5. Evaluates the model at both default and tuned thresholds
# MAGIC 6. Writes predictions back to a gold table
# MAGIC 7. Logs and registers the model in MLflow / Unity Catalog
# MAGIC
# MAGIC ---
# MAGIC **⚠️ Before running:** confirm that `Previous_Amount`, `Amount_Difference`, `Avg_Last10`, and
# MAGIC `Customer_Total_Transactions` in `transaction_features` are computed point-in-time
# MAGIC (using only prior transactions). If these were built with a window that includes the
# MAGIC current or future transaction, the metrics below will be optimistic and will not hold up
# MAGIC in production.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Setup & Imports

# COMMAND ----------

import pandas as pd
import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)
from xgboost import XGBClassifier

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Load Data
# MAGIC
# MAGIC Reads transaction features from the gold table and checks the class balance
# MAGIC of `Is_Fraud` before doing anything else — fraud datasets are typically heavily
# MAGIC imbalanced, which drives several modeling choices further down (`scale_pos_weight`,
# MAGIC `aucpr` as the eval metric, stratified splits).

# COMMAND ----------

df = spark.table("ujjivan_2.gold.transaction_features")

df.groupBy("Is_Fraud").count().orderBy("Is_Fraud").show()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2a. Inspect schema
# MAGIC
# MAGIC The `_idx` columns referenced later (`Gender_idx`, `State_idx`, etc.) are **not**
# MAGIC present in the raw table — only the original categorical columns are (e.g. `Gender`,
# MAGIC `District`). Print the schema below and confirm the actual categorical column names
# MAGIC before relying on the `categorical_columns` list further down — in particular, check
# MAGIC whether your table uses `State`/`City` or a single `District` column instead.

# COMMAND ----------

df.printSchema()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2b. Index categorical columns
# MAGIC
# MAGIC Builds the `_idx` columns via `StringIndexer` so downstream feature selection works.
# MAGIC **Update `categorical_columns` below to match what you saw in `printSchema()` above** —
# MAGIC this assumes `State` and `City` exist; if your table only has `District`, adjust
# MAGIC accordingly (and update `feature_columns` later in the notebook to match).

# COMMAND ----------

from pyspark.ml.feature import StringIndexer

categorical_columns = [
    "Gender",
    "State",
    "City",
    "Account_Type",
    "Merchant_Category",
    "Transaction_Type",
    "Device_Type"
]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_columns
]

for indexer in indexers:
    df = indexer.fit(df).transform(df)

print("Indexed columns added:", [f"{c}_idx" for c in categorical_columns])

# COMMAND ----------

feature_columns = [
    "Age",
    "Transaction_Amount",
    "Account_Balance",
    "Hour",
    "DayOfWeek",
    "Month",
    "Weekend",
    "HighValue",
    "DigitalTransaction",
    "Previous_Amount",
    "Amount_Difference",
    "Avg_Last10",
    "Customer_Total_Transactions",
    "Merchant_Diversity",
    "Gender_idx",
    "State_idx",
    "City_idx",
    "Account_Type_idx",
    "Merchant_Category_idx",
    "Transaction_Type_idx",
    "Device_Type_idx"
]

target_column = "Is_Fraud"

ml_df = df.select(feature_columns + [target_column])

print("Total records:", ml_df.count())

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Convert to Pandas
# MAGIC
# MAGIC `toPandas()` pulls the full dataset into driver memory. Fine for datasets up to a
# MAGIC few hundred thousand rows on a reasonably sized driver — if `transaction_features`
# MAGIC grows into the tens of millions of rows, consider sampling or switching to a
# MAGIC distributed training approach (e.g. `xgboost.spark`) instead.

# COMMAND ----------

pdf = ml_df.toPandas()
print("Pulled shape:", pdf.shape)

pdf = pdf.replace([float("inf"), float("-inf")], pd.NA)
pdf = pdf.fillna(0)

X = pdf[feature_columns]
y = pdf[target_column]

print("Features:", X.shape)
print("Target:", y.shape)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Train / Validation / Test Split
# MAGIC
# MAGIC Three-way split so that `X_test` stays completely untouched until final evaluation.
# MAGIC The validation set is used only for early stopping during training.
# MAGIC All splits are stratified on `Is_Fraud` to preserve the class ratio.

# COMMAND ----------

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.15,
    random_state=42,
    stratify=y_train_full
)

print("Training  :", X_train.shape)
print("Validation:", X_val.shape)
print("Testing   :", X_test.shape)

# COMMAND ----------

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()
scale_pos_weight = negative / positive

print("Normal transactions:", negative)
print("Fraud transactions :", positive)
print("scale_pos_weight    :", scale_pos_weight)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Train Model
# MAGIC
# MAGIC XGBoost classifier with:
# MAGIC - `scale_pos_weight` to rebalance the loss for the minority (fraud) class
# MAGIC - `aucpr` as the eval metric, which is more informative than accuracy or plain AUC
# MAGIC   on imbalanced data
# MAGIC - Early stopping on the validation set, so we don't overfit a fixed number of
# MAGIC   boosting rounds

# COMMAND ----------

fraud_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1
)

fraud_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Best iteration:", fraud_model.best_iteration)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Threshold Selection
# MAGIC
# MAGIC The default 0.5 cutoff from `.predict()` is arbitrary for a rebalanced fraud model.
# MAGIC Instead, we pick a threshold that targets a specific recall level — i.e. "catch at
# MAGIC least X% of fraud" — since that's usually how the business frames the tradeoff.
# MAGIC
# MAGIC **Adjust `target_recall` below based on what your fraud/compliance team can tolerate
# MAGIC in terms of false positives.** A higher recall target will flag more legitimate
# MAGIC transactions as fraud; a lower one will miss more fraud.

# COMMAND ----------

y_probability = fraud_model.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_probability)

target_recall = 0.80  # <-- adjust based on business tolerance
valid_idx = [i for i, r in enumerate(recalls[:-1]) if r >= target_recall]
chosen_threshold = thresholds[valid_idx[-1]] if valid_idx else 0.5

print(f"Chosen threshold for recall >= {target_recall}:", chosen_threshold)

y_pred_default = (y_probability >= 0.5).astype(int)
y_pred_tuned = (y_probability >= chosen_threshold).astype(int)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Evaluation
# MAGIC
# MAGIC Comparing the default 0.5 threshold against the tuned, recall-targeted threshold
# MAGIC side by side, along with the threshold-independent ROC-AUC and PR-AUC scores.

# COMMAND ----------

print("\n--- Default threshold (0.5) ---")
print(classification_report(y_test, y_pred_default))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_default))

print("\n--- Tuned threshold ---")
print(classification_report(y_test, y_pred_tuned))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

# COMMAND ----------

roc_auc = roc_auc_score(y_test, y_probability)
pr_auc = average_precision_score(y_test, y_probability)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Write Predictions Back to Gold
# MAGIC
# MAGIC Persists test-set predictions (both threshold variants and raw probability) as a
# MAGIC gold table for downstream review — e.g. by a fraud analyst dashboard or BI tool.

# COMMAND ----------

prediction_pdf = X_test.copy()
prediction_pdf["Actual_Is_Fraud"] = y_test.values
prediction_pdf["Predicted_Is_Fraud_Default"] = y_pred_default
prediction_pdf["Predicted_Is_Fraud_Tuned"] = y_pred_tuned
prediction_pdf["Fraud_Probability"] = y_probability

spark.createDataFrame(prediction_pdf) \
    .write.mode("overwrite") \
    .saveAsTable("ujjivan_2.gold.fraud_predictions")

print("Predictions written to ujjivan_2.gold.fraud_predictions")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. MLflow Logging & Model Registration
# MAGIC
# MAGIC Logs params, metrics, and the model itself, with a signature and input example so
# MAGIC it's ready for Databricks model serving. Registers to Unity Catalog if available —
# MAGIC remove `mlflow.set_registry_uri("databricks-uc")` and `registered_model_name` if
# MAGIC your workspace isn't on UC.
# MAGIC
# MAGIC **Update `registered_model_name` below to match your catalog/schema naming
# MAGIC convention before running.**

# COMMAND ----------

mlflow.set_registry_uri("databricks-uc")  # remove this line if not on Unity Catalog

signature = infer_signature(X_train, fraud_model.predict(X_train))

with mlflow.start_run(run_name="bank_fraud_xgboost") as run:

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("scale_pos_weight", scale_pos_weight)
    mlflow.log_param("best_iteration", fraud_model.best_iteration)
    mlflow.log_param("chosen_threshold", chosen_threshold)

    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)

    mlflow.xgboost.log_model(
        fraud_model,
        "fraud_xgboost_model",
        signature=signature,
        input_example=X_train.iloc[:5],
        registered_model_name="ujjivan_2.gold.bank_fraud_xgboost"  # update catalog.schema.model_name as needed
    )

    print("MLflow run ID:", run.info.run_id)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 10. Next Steps
# MAGIC
# MAGIC - Validate feature point-in-time correctness upstream (see warning at top of notebook)
# MAGIC - Review `ujjivan_2.gold.fraud_predictions` with the fraud/risk team to sanity-check
# MAGIC   false positives at the tuned threshold
# MAGIC - If satisfied, promote the registered model version to a serving endpoint
# MAGIC - Consider SHAP-based feature importance for model explainability, especially if
# MAGIC   this feeds into any regulatory or audit process